03/03/2026

Building an Interactive AI Agent

-short term memory (history prompt)

-long term memory (Vector DB)

-User Preferences (VDB)

-execution (function calling | tools)

In [1]:
import os
import json
from dotenv import load_dotenv
from ollama import Client

load_dotenv()

# ==========================================
# 1. SETUP THE CLIENT
# ==========================================
api_key = os.getenv("OLLAMA_API_KEY") 
client = Client(
    host='https://ollama.com',
    headers={'Authorization': f'Bearer {api_key}'}
)

# ==========================================
# 2. DEFINE YOUR TOOLS (The VTU Actions)
# ==========================================
def buy_airtime(phone_number: str, amount: int, network: str) -> str:
    """Purchases airtime for a specific phone number and telecom network.
    
    Args:
        phone_number: The 11-digit phone number (e.g., '08012345678').
        amount: The amount of Naira to recharge.
        network: The telecom network (e.g., 'MTN', 'Airtel', 'Glo', '9mobile').
    """
    print(f"\n⚡ [BACKEND ACTION]: Processing N{amount} {network} airtime for {phone_number}...")
    return f"Transaction Successful: N{amount} airtime sent to {phone_number} on {network}."

def check_balance() -> str:
    """Checks the user's current wallet balance."""
    print("\n⚡ [BACKEND ACTION]: Querying database for wallet balance...")
    return "The current wallet balance is N5,500."

# We pass these functions to Ollama
vtu_tools = [buy_airtime, check_balance]

# ==========================================
# 3. INITIALIZE MEMORY & INSTRUCTIONS
# ==========================================
# Notice how much shorter the prompt is now that tools handle the heavy lifting!
system_instruction = """
You are Topify AI, an interactive and helpful Virtual Top-Up (VTU) assistant. 
Your job is to help the user buy airtime, buy data, and manage their wallet.
- Be conversational, concise, and friendly.
- If the user wants to do something you have a tool for, use the tool.
- If details are missing (like network or amount), politely ask the user for them.
"""

# This list IS our short-term memory. We start it with the system rules.
conversation_history = [
    {'role': 'system', 'content': system_instruction}
]

# ==========================================
# 4. START THE CHAT LOOP
# ==========================================
print("Starting Topify AI (Ollama Edition)... (Type 'quit' to exit)\n")

# Use the model from your snippet. If it struggles with tools, you might want to 
# swap this out for a strong open-source tool-calling model from the Qwen series.
MODEL_NAME = 'gpt-oss:20b-cloud' 

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit']:
        break

    # 1. Add user message to memory
    conversation_history.append({'role': 'user', 'content': user_input})

    # 2. Send the entire memory and tools to Ollama
    response = client.chat(
        model=MODEL_NAME,
        messages=conversation_history,
        tools=vtu_tools
    )

    # 3. Check if the model decided to use a tool
    if response.message.tool_calls:
        for tool in response.message.tool_calls:
            # The model tells us which function to run and gives us the arguments
            function_name = tool.function.name
            arguments = tool.function.arguments
            
            # Add the model's tool request to memory
            conversation_history.append(response.message)
            
            # Execute the actual Python function based on what the model requested
            if function_name == 'buy_airtime':
                tool_result = buy_airtime(**arguments)
            elif function_name == 'check_balance':
                tool_result = check_balance()
            else:
                tool_result = "Error: Unknown function."
                
            # Add the function's output back into memory so the model can read it
            conversation_history.append({
                'role': 'tool',
                'content': tool_result,
                'name': function_name
            })
            
            # Ask the model to generate a final conversational reply based on the tool result
            final_response = client.chat(model=MODEL_NAME, messages=conversation_history)
            
            print(f"Topify AI: {final_response.message.content}\n")
            conversation_history.append(final_response.message)
            
    else:
        # If no tool was called, just print the standard conversational response
        print(f"Topify AI: {response.message.content}\n")
        conversation_history.append(response.message)

Starting Topify AI (Ollama Edition)... (Type 'quit' to exit)

Topify AI: Hello! How can I help you today? Would you like to buy airtime, purchase data, or check your wallet balance? Just let me know!

Topify AI: Sure thing! Let me know what you’d like to do—buy airtime, buy data, or check your wallet balance—and I’ll guide you through it. If you’re unsure, I can give you a quick overview of the options.



KeyboardInterrupt: Interrupted by user